# Algoritmos de clasificación con Scikit-Learn
Informe de Machine Learning Supervisado - Notebook 2

Decision Tree, Random Forest, SVM, KNN y Naive Bayes (con Regresión Logística como referencia).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target          # 0 = maligno, 1 = benigno
print(X.shape)
print(y.map({0: "Maligno", 1: "Benigno"}).value_counts())
X.head()

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(Xtr.shape, Xte.shape)

## 1. Definición de modelos
SVM, KNN y Regresión Logística usan estandarización porque dependen de distancias o de la escala de las variables; los modelos basados en árboles y Naive Bayes no la requieren.

In [ ]:
modelos = {
    "Regresión Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=42)),
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
}

## 2. Entrenamiento y evaluación

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
filas, matrices, probs = [], {}, {}
for nombre, m in modelos.items():
    m.fit(Xtr, ytr)
    p = m.predict(Xte); pr = m.predict_proba(Xte)[:, 1]
    matrices[nombre], probs[nombre] = confusion_matrix(yte, p), pr
    cvs = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    filas.append({"Modelo": nombre, "Exactitud": accuracy_score(yte, p), "Precisión": precision_score(yte, p),
                  "Recall": recall_score(yte, p), "F1": f1_score(yte, p), "AUC": roc_auc_score(yte, pr),
                  "CV media": cvs.mean(), "CV desv.": cvs.std()})
resultados = pd.DataFrame(filas).round(4)
resultados

## 3. Matrices de confusión

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10, 6.4))
for ax, (n, c) in zip(axs.ravel(), matrices.items()):
    ConfusionMatrixDisplay(c, display_labels=["Maligno", "Benigno"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(n, fontsize=10)
plt.tight_layout(); plt.show()

## 4. Curvas ROC

In [ ]:
plt.figure(figsize=(6, 4.6))
for n, pr in probs.items():
    f, t, _ = roc_curve(yte, pr)
    plt.plot(f, t, label=f"{n} (AUC={roc_auc_score(yte, pr):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=.8)
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Tasa de verdaderos positivos"); plt.legend(fontsize=7); plt.show()

## 5. Decision Tree: estructura

In [ ]:
plt.figure(figsize=(11, 5))
plot_tree(modelos["Decision Tree"], max_depth=2, feature_names=X.columns, class_names=["Maligno", "Benigno"], filled=True, fontsize=7)
plt.show()

## 6. Random Forest: importancia de variables

In [ ]:
imp = pd.Series(modelos["Random Forest"].feature_importances_, index=X.columns).sort_values().tail(10)
imp.plot.barh(figsize=(6, 3.6)); plt.title("10 variables más importantes"); plt.tight_layout(); plt.show()

## 7. KNN: efecto de k

In [ ]:
ks = range(1, 26)
accs = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), X, y, cv=cv).mean() for k in ks]
plt.plot(list(ks), accs, marker="o", ms=3); plt.xlabel("k"); plt.ylabel("Exactitud (CV)"); plt.show()
print("Mejor k:", list(ks)[int(np.argmax(accs))])